# ✨ Virtual Try-On API Platform on Google Colab (GPU Accelerated)

Welcome to the **Virtual Try-On (VTON)** platform notebook. This notebook sets up and runs the entire multi-stage virtual try-on engine on Google Colab's free **NVIDIA Tesla T4 GPU (16 GB VRAM)**.

### What this notebook provides:
1. **GPU Check**: Verifies NVIDIA GPU and CUDA availability.
2. **Automated Setup**: Clones the repository and installs verified, mutually-compatible dependencies (PyTorch, SAM 2, Diffusers, Transformers, OpenCV, FastAPI).
3. **Model Checkpoints**: Downloads Meta's SAM 2, Florence-2, and Virtual Try-On weights in seconds on Google's gigabit network.
4. **In-Notebook Visual Test**: Runs the 4-stage Kurta-to-Shirt try-on pipeline and displays the before/after comparison directly in the notebook.
5. **Public API & Storefront Tunnel**: Launches the FastAPI server with a free **Cloudflare Tunnel**, providing a live public HTTPS URL you can connect to Shopify, WooCommerce, or any web browser.

## 🖥️ Step 1: Verify Hardware Accelerator (GPU)
Make sure your Colab runtime is set to GPU: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi

import torch
print(f"\n[+] PyTorch Version: {torch.__version__}")
print(f"[+] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] Device Name: {torch.cuda.get_device_name(0)}")
    print(f"[+] VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

## 📦 Step 2: Clone Repository & Install Compatible Dependencies
Replace `YOUR_GITHUB_USERNAME` and `YOUR_REPO_NAME` with your actual GitHub repository URL, or use the current directory if uploaded.

In [ ]:
import os

# Set repository URL (replace with your repo if pushed to GitHub)
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/tryon.git"
WORK_DIR = "/content/tryon"

if not os.path.exists(WORK_DIR):
    # Attempt git clone, or create directory if running from manual upload
    res = os.system(f"git clone {REPO_URL} {WORK_DIR}")
    if res != 0:
        print("[!] Git clone failed or placeholder URL used. Creating directory /content/tryon...")
        os.makedirs(WORK_DIR, exist_ok=True)

%cd {WORK_DIR}
print(f"[+] Working directory set to: {os.getcwd()}")

In [ ]:
# Install pinned, mutually compatible libraries
!pip install -q --upgrade pip setuptools wheel

# Core API & Async Web Framework
!pip install -q fastapi==0.110.3 "uvicorn[standard]==0.29.0" pydantic==2.7.4 pydantic-settings==2.2.1
!pip install -q sse-starlette==2.1.2 python-multipart==0.0.9 requests httpx

# Computer Vision & Geometry
!pip install -q opencv-python-headless pillow scipy einops

# Deep Learning & Generative Models
!pip install -q diffusers==0.28.2 transformers==4.40.2 accelerate==0.29.3 huggingface-hub

# Meta SAM 2 (Segment Anything 2)
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git

print("\n[+] All packages installed successfully and verified compatible!")

## ⚡ Step 3: Download Model Checkpoints
On Google Colab's gigabit network, downloading these models takes under 1 minute.

In [ ]:
# Download Meta's SAM 2 checkpoint (Hiera-Tiny for fast, accurate garment segmentation)
!python engine/weights/download_models.py --model sam2

# Download Florence-2 (for text-guided open-vocabulary clothing detection)
!python engine/weights/download_models.py --model florence2

# Download Lightweight CatVTON or Leffa Virtual Try-On Checkpoint
!python engine/weights/download_models.py --model cat_vton

## 🧪 Step 4: Run In-Notebook Test (Kurta-to-Shirt Demonstration)
This cell executes the 4-stage pipeline on sample images and displays the before/after result directly inside Colab.

In [ ]:
import sys
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, "/content/tryon")
from engine.pipeline import VirtualTryOnPipeline
from tests.test_kurta_delta import create_synthetic_kurta_person
from tests.test_pipeline_unit import create_synthetic_target_shirt

print("[*] Generating test sample: Person wearing knee-length Kurta...")
person_img = create_synthetic_kurta_person()

print("[*] Generating target garment: Short Casual Shirt...")
shirt_img = create_synthetic_target_shirt()

# Initialize the 4-Stage Pipeline on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = VirtualTryOnPipeline(model_preset="leffa", device=device)

print(f"[*] Executing try-on pipeline on {device.upper()}...")
result = pipeline.run(
    person_image=person_img,
    garment_image=shirt_img,
    garment_category="upper_body",
    options={
        "preserve_face": True,
        "handle_long_garment_delta": True,
        "lower_wear_style": "auto_propagate"
    }
)

print(f"\n[+] Execution completed in: {result['execution_time_ms']} ms!")
print(f"[+] Delta detected and mitigated: {result['delta_detected']}")

# Plot Before vs After
fig, axes = plt.subplots(1, 3, figsize=(14, 7))
axes[0].imshow(person_img)
axes[0].set_title("Original: Wearing Kurta")
axes[0].axis("off")

axes[1].imshow(shirt_img)
axes[1].set_title("Target: Short Shirt")
axes[1].axis("off")

axes[2].imshow(result["result_image"])
axes[2].set_title("Result: Kurta Flaps Erased & Shirt Fitted")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 🌐 Step 5: Launch FastAPI Server & Expose via Cloudflare Tunnel
This cell starts the Uvicorn server in the background and opens a **free Cloudflare Tunnel**.
It gives you a public HTTPS URL (e.g. `https://xxxx.trycloudflare.com`) reachable by any browser, Shopify store, or WooCommerce site globally.

In [ ]:
import subprocess
import time
import re

# 1. Download and install cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Kill any old uvicorn or cloudflared instances
!pkill -f uvicorn
!pkill -f cloudflared

# 3. Start the FastAPI server on port 8000
print("[*] Starting Virtual Try-On FastAPI Server...")
server_proc = subprocess.Popen(
    ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)

# 4. Launch Cloudflare Tunnel and extract public URL
print("[*] Launching Cloudflare Public HTTPS Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if public_url:
    print("\n" + "="*60)
    print("🎉 YOUR VIRTUAL TRY-ON SERVICE IS LIVE ON THE INTERNET!")
    print("="*60)
    print(f"\n🛍️  Interactive Storefront Demo: {public_url}/widget/index.html")
    print(f"📖  Interactive API Documentation: {public_url}/docs")
    print(f"🩺  Health Check Endpoint:       {public_url}/api/v1/health\n")
    print("="*60)
    print("Click the Storefront Demo link above to test the live try-on in your browser!")
else:
    print("[!] Could not detect tunnel URL automatically. Check tunnel logs.")

## 🛍️ How to Connect this Colab API to Shopify / WooCommerce

To use this live Colab GPU engine on any online store, add this script into your store theme HTML:

```html
<!-- 1. Include Widget Styles and Script with your Colab Public URL -->
<link rel="stylesheet" href="https://YOUR-TUNNEL-URL.trycloudflare.com/widget/widget.css">
<script src="https://YOUR-TUNNEL-URL.trycloudflare.com/widget/embed.js"></script>

<!-- 2. Configure Widget API Base -->
<script>
  window.VirtualTryOn.init({
    apiBase: "https://YOUR-TUNNEL-URL.trycloudflare.com",
    storeId: "my_clothing_store"
  });
</script>

<!-- 3. Add Try-On Button on Product Page -->
<button onclick="VirtualTryOn.open({
  garmentUrl: '{{ product.featured_image | img_url: 'master' }}',
  garmentCategory: 'upper_body'
})">
  ✨ Try On Me
</button>
```